In [1]:
import sys
!{sys.executable} -m pip install pygam


   ---------------------------------------- 0/3 [python-utils]
   ---------------------------------------- 0/3 [python-utils]
   ------------- -------------------------- 1/3 [progressbar2]
   ------------- -------------------------- 1/3 [progressbar2]
   ------------- -------------------------- 1/3 [progressbar2]
   ------------- -------------------------- 1/3 [progressbar2]
   ------------- -------------------------- 1/3 [progressbar2]
   -------------------------- ------------- 2/3 [pygam]
   -------------------------- ------------- 2/3 [pygam]
   -------------------------- ------------- 2/3 [pygam]
   -------------------------- ------------- 2/3 [pygam]
   -------------------------- ------------- 2/3 [pygam]
   ---------------------------------------- 3/3 [pygam]




[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score, roc_curve
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('Imports Parte A OK')
try:
    from pygam import LogisticGAM, s, l, f
    PYGAM_OK = True
    print('pyGAM disponible ✓')
except ImportError:
    PYGAM_OK = False
    print('pyGAM no instalado — instalar con: pip install pygam')
    print('La Parte B se saltará si pyGAM no está disponible.')

Imports Parte A OK
pyGAM disponible ✓


In [3]:
SEED    = 42
N_TRIALS = 20
np.random.seed(SEED)

BASE_DIR = Path('../../')
DATA_DIR = BASE_DIR / 'data/processed'
FIG_DIR  = BASE_DIR / 'paper/output/figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

matplotlib.rcParams.update({'figure.figsize': (9, 6), 'font.size': 12,
                            'axes.spines.top': False, 'axes.spines.right': False})

In [4]:
df      = pd.read_parquet(DATA_DIR / 'features_train.parquet')
df_test = pd.read_parquet(DATA_DIR / 'features_test.parquet')

feature_cols = [c for c in df.columns if c not in ('SK_ID_CURR', 'TARGET')]

X_raw       = df[feature_cols].values
y           = df['TARGET'].values
X_test_raw  = df_test[feature_cols].values
sk_ids_test = df_test['SK_ID_CURR'].values

print(f'Train: {X_raw.shape}  | Default rate: {y.mean():.2%}')

# Identificar features binarias (≤ 2 valores únicos después de imputar)
# Para no aplicar splines a indicadores 0/1
n_unique = np.array([len(np.unique(col[~np.isnan(col)])) for col in X_raw.T])
binary_mask   = n_unique <= 2
continuous_mask = ~binary_mask

binary_feats     = [feature_cols[i] for i in range(len(feature_cols)) if binary_mask[i]]
continuous_feats = [feature_cols[i] for i in range(len(feature_cols)) if continuous_mask[i]]

print(f'Features binarias   ({len(binary_feats)}): {binary_feats}')
print(f'Features continuas ({len(continuous_feats)}): {continuous_feats[:5]}...')

Train: (307511, 30)  | Default rate: 8.07%
Features binarias   (1): ['CODE_GENDER']
Features continuas (29): ['CREDIT_ANNUITY_RATIO', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH']...


In [5]:
def compute_metrics(y_true, y_prob, threshold=0.5, label='Model'):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    auc  = roc_auc_score(y_true, y_prob)
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(Model=label, AUC=round(auc,4),
                N=len(y_true), P=int(y_pred.sum()),
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                Recall=round(rec,4), Precision=round(prec,4), F1=round(f1,4))

def plot_roc_curve(y_true, y_prob, label, color, ax, title='ROC Curve'):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC = {auc:.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(title); ax.legend()
    return ax

print('Funciones definidas ✓')

Funciones definidas ✓


---
## PARTE A — SplineTransformer + Logit L1

**Equivalente Python de `bs() + glmnet` en R.**  
Se aplica `SplineTransformer(degree=3)` a las features continuas (B-splines cúbicos)
dejando las binarias sin transformar. Luego Logit con L1.

Optuna optimiza: `n_knots ∈ [3, 8]` y `C ∈ [1e-4, 10]`.

In [6]:
from sklearn.compose import ColumnTransformer

# Indices de features continuas y binarias en el array
cont_idx = [i for i, c in enumerate(feature_cols) if c in continuous_feats]
bin_idx  = [i for i, c in enumerate(feature_cols) if c in binary_feats]

imputer = SimpleImputer(strategy='median')
X_imp   = imputer.fit_transform(X_raw)
X_test_imp = imputer.transform(X_test_raw)

print(f'X_imp shape: {X_imp.shape} — NaNs: {np.isnan(X_imp).sum()}')
print(f'Índices continuos: {cont_idx[:5]}... | Índices binarios: {bin_idx}')

X_imp shape: (307511, 30) — NaNs: 0
Índices continuos: [0, 1, 2, 3, 4]... | Índices binarios: [27]


In [7]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def make_spline_pipeline(n_knots, C):
    """Pipeline: splines en continuas + passthrough en binarias + logit L1."""
    ct = ColumnTransformer([
        ('splines', Pipeline([
            ('sp', SplineTransformer(n_knots=n_knots, degree=3, include_bias=False)),
            ('sc', StandardScaler())
        ]), cont_idx),
        ('binary', StandardScaler(), bin_idx)
    ])
    lr = LogisticRegression(penalty='l1', C=C, solver='saga',
                            class_weight='balanced', max_iter=2000,
                            random_state=SEED)
    return Pipeline([('preproc', ct), ('lr', lr)])

def objective_a(trial):
    n_knots = trial.suggest_int('n_knots', 3, 8)
    C       = trial.suggest_float('C', 1e-4, 10.0, log=True)

    pipe = make_spline_pipeline(n_knots, C)
    cv_auc = cross_val_score(pipe, X_imp, y, cv=skf,
                             scoring='roc_auc', n_jobs=-1).mean()

    # Penalizar overfitting: train AUC vs CV AUC
    pipe.fit(X_imp, y)
    train_auc = roc_auc_score(y, pipe.predict_proba(X_imp)[:, 1])
    gap = max(0, train_auc - cv_auc)
    return cv_auc - 0.5 * gap

print(f'Lanzando Optuna Parte A — {N_TRIALS} trials...')
study_a = optuna.create_study(direction='maximize',
                               sampler=optuna.samplers.TPESampler(seed=SEED))
study_a.optimize(objective_a, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nMejores hiperparámetros (Parte A):')
print(f'  n_knots : {study_a.best_params["n_knots"]}')
print(f'  C       : {study_a.best_params["C"]:.6f}')
print(f'  Objetivo: {study_a.best_value:.4f}')

Lanzando Optuna Parte A — 20 trials...


  0%|          | 0/20 [00:00<?, ?it/s]

[W 2026-02-23 19:44:46,123] Trial 0 failed with parameters: {'n_knots': 5, 'C': 5.669849511478847} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\HP\AppData\Local\Temp\ipykernel_10892\2743159863.py", line 22, in objective_a
    cv_auc = cross_val_score(pipe, X_imp, y, cv=skf,
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 677, in cross_val_score
    cv_results = cross_validate(
                 ^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [ ]:
# Modelo final Parte A — refit en todo el train
best_p_a = study_a.best_params
pipe_a_final = make_spline_pipeline(best_p_a['n_knots'], best_p_a['C'])
pipe_a_final.fit(X_imp, y)

y_prob_a = pipe_a_final.predict_proba(X_imp)[:, 1]
metrics_a = compute_metrics(y, y_prob_a, threshold=0.5, label='SplineLogit (A)')
metrics_a['CV_AUC'] = round(study_a.best_value, 4)

print('PARTE A — SplineTransformer + Logit L1:')
for k, v in metrics_a.items():
    if k != 'Model':
        print(f'  {k:<12}: {v}')

In [ ]:
# Optuna: historial de trials
trial_df = study_a.trials_dataframe()[['number','value','params_n_knots','params_C']]
trial_df.columns = ['Trial','Objetivo','n_knots','C']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(trial_df['Trial'], trial_df['Objetivo'], 'o-', color='#3498db')
axes[0].axhline(study_a.best_value, color='#e74c3c', ls='--', label=f'Best={study_a.best_value:.4f}')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Objetivo (CV AUC penalizado)')
axes[0].set_title('Optuna — Parte A'); axes[0].legend()

sc = axes[1].scatter(np.log10(trial_df['C']), trial_df['n_knots'],
                     c=trial_df['Objetivo'], cmap='RdYlGn', s=80)
plt.colorbar(sc, ax=axes[1], label='Objetivo')
axes[1].set_xlabel('log10(C)'); axes[1].set_ylabel('n_knots')
axes[1].set_title('Espacio de hiperparámetros — Parte A')

plt.tight_layout()
fig.savefig(FIG_DIR / '08_optuna_parte_a.pdf', bbox_inches='tight')
plt.show()
print('Figura guardada: 08_optuna_parte_a.pdf')

---
## PARTE B — pyGAM (LogisticGAM)

GAM formal con términos suaves `s()` para features continuas y términos lineales `l()`
para binarias. La selección de parámetro de suavizado se hace vía **GCV** con `.gridsearch()`.

Diferencia clave con Parte A: cada variable tiene su propio λ (suavizado independiente).

In [ ]:
if not PYGAM_OK:
    print('pyGAM no disponible. Instalar con: pip install pygam')
    print('Saltando Parte B.')
else:
    # pyGAM necesita array sin NaN → usar X_imp
    # Escalar antes de pasar a GAM (mejora convergencia)
    scaler_b = StandardScaler()
    X_scaled_b      = scaler_b.fit_transform(X_imp)
    X_test_scaled_b = scaler_b.transform(X_test_imp)

    # Construir términos del GAM:
    # s(i) para continuas, l(i) para binarias
    terms_list = []
    for i, feat in enumerate(feature_cols):
        if feat in binary_feats:
            terms_list.append(l(i))   # término lineal
        else:
            terms_list.append(s(i, n_splines=10, spline_order=3))  # spline cúbico

    from pygam import te
    terms_formula = terms_list[0]
    for t in terms_list[1:]:
        terms_formula = terms_formula + t

    print(f'Términos GAM: {len(terms_list)} ({sum(1 for f in feature_cols if f not in binary_feats)} splines + {len(binary_feats)} lineales)')

    # Usar subsample para gridsearch (300k filas puede ser lento)
    N_SAMPLE_GAM = 50_000
    rng = np.random.default_rng(SEED)
    idx_sample = rng.choice(len(y), size=N_SAMPLE_GAM, replace=False)
    X_gam_s = X_scaled_b[idx_sample]
    y_gam_s = y[idx_sample]

    print(f'Subsample para gridsearch: {N_SAMPLE_GAM:,} filas')
    print(f'Default rate subsample: {y_gam_s.mean():.2%}')

In [ ]:
if PYGAM_OK:
    print('Ajustando LogisticGAM con gridsearch (GCV)...')
    gam = LogisticGAM(terms_formula).fit(X_gam_s, y_gam_s)

    # Gridsearch sobre lam (parámetro de suavizado global)
    lam_grid = np.logspace(-3, 3, 10)
    gam_gs = gam.gridsearch(X_gam_s, y_gam_s,
                            lam=lam_grid,
                            progress=True,
                            keep_best=True)

    print(f'\nMejor lambda (GCV): {gam_gs.lam}')
    print(f'GCV score: {gam_gs.statistics_["GCV"]:.4f}')
else:
    gam_gs = None

In [ ]:
if PYGAM_OK and gam_gs is not None:
    # Refit en dataset completo con el mejor lambda encontrado
    print('Refitteando GAM en dataset completo...')
    gam_final = LogisticGAM(terms_formula, lam=gam_gs.lam)
    gam_final.fit(X_scaled_b, y)

    y_prob_b = gam_final.predict_proba(X_scaled_b)

    # CV AUC del GAM (usando el subsample → CV aproximado)
    from sklearn.model_selection import cross_val_score as cvs_sk
    # pyGAM no tiene interfaz sklearn directa, computar manualmente con skf pequeño
    gam_cv_aucs = []
    skf_gam = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    for tr_i, val_i in skf_gam.split(X_gam_s, y_gam_s):
        g = LogisticGAM(terms_formula, lam=gam_gs.lam).fit(X_gam_s[tr_i], y_gam_s[tr_i])
        prob = g.predict_proba(X_gam_s[val_i])
        gam_cv_aucs.append(roc_auc_score(y_gam_s[val_i], prob))

    gam_cv_mean = np.mean(gam_cv_aucs)
    print(f'CV AUC GAM (3-fold, subsample): {gam_cv_mean:.4f}')

    metrics_b = compute_metrics(y, y_prob_b, threshold=0.5, label='pyGAM (B)')
    metrics_b['CV_AUC'] = round(gam_cv_mean, 4)

    print('\nPARTE B — pyGAM:')
    for k, v in metrics_b.items():
        if k != 'Model':
            print(f'  {k:<12}: {v}')

In [ ]:
if PYGAM_OK and gam_gs is not None:
    # Partial dependence plots (top 6 features más suaves)
    top_cont = [i for i, f in enumerate(feature_cols) if f in continuous_feats[:6]]
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.ravel()
    for j, idx_f in enumerate(top_cont[:6]):
        XX = gam_final.generate_X_grid(term=j)
        pdep, confi = gam_final.partial_dependence(term=j, X=XX, width=0.95)
        axes[j].plot(XX[:, idx_f], pdep, color='#e74c3c', lw=2)
        axes[j].fill_between(XX[:, idx_f], confi[:, 0], confi[:, 1],
                             alpha=0.2, color='#e74c3c')
        axes[j].set_title(feature_cols[idx_f])
        axes[j].set_xlabel('Valor'); axes[j].set_ylabel('Partial dependence')

    plt.suptitle('Partial Dependence Plots — pyGAM', fontsize=13)
    plt.tight_layout()
    fig.savefig(FIG_DIR / '08_partial_dependence_gam.pdf', bbox_inches='tight')
    plt.show()
    print('Figura guardada: 08_partial_dependence_gam.pdf')

---
## Comparación Parte A vs Parte B

In [ ]:
results_list = [metrics_a]
if PYGAM_OK and gam_gs is not None:
    results_list.append(metrics_b)

compare_df = pd.DataFrame(results_list).set_index('Model')
display(compare_df)

# ROC comparativa
fig, ax = plt.subplots(figsize=(8, 6))
plot_roc_curve(y, y_prob_a, label='SplineLogit (A)', color='#3498db', ax=ax,
               title='ROC — Splines vs GAM')
if PYGAM_OK and gam_gs is not None:
    plot_roc_curve(y, y_prob_b, label='pyGAM (B)', color='#e74c3c', ax=ax,
                   title='ROC — Splines vs GAM')
plt.tight_layout()
fig.savefig(FIG_DIR / '08_roc_splines_vs_gam.pdf', bbox_inches='tight')
plt.show()
print('Figura guardada: 08_roc_splines_vs_gam.pdf')

## Predicciones Test

In [ ]:
# Parte A — predicciones test
y_pred_test_a = pipe_a_final.predict_proba(X_test_imp)[:, 1]
sub_a = pd.DataFrame({'SK_ID_CURR': sk_ids_test, 'TARGET': y_pred_test_a})
_sub_path_a = DATA_DIR / 'submission_08a_spline_logit.csv'
sub_a.to_csv(_sub_path_a, index=False)
print(f'Submission A guardada: {_sub_path_a.name}  ({len(sub_a):,} filas)')

# Parte B — predicciones test (si pyGAM disponible)
if PYGAM_OK and gam_gs is not None:
    y_pred_test_b = gam_final.predict_proba(X_test_scaled_b)
    sub_b = pd.DataFrame({'SK_ID_CURR': sk_ids_test, 'TARGET': y_pred_test_b})
    _sub_path_b = DATA_DIR / 'submission_08b_pygam.csv'
    sub_b.to_csv(_sub_path_b, index=False)
    print(f'Submission B guardada: {_sub_path_b.name}  ({len(sub_b):,} filas)')
else:
    print('Submission B omitida (pyGAM no disponible)')

## Resumen Final

In [ ]:
print('=' * 65)
print('LOGIT CON SPLINES — RESUMEN COMPARATIVO')
print('=' * 65)
print(f'Parte A (SplineTransformer + L1):')
print(f'  n_knots  = {best_p_a["n_knots"]} | C = {best_p_a["C"]:.5f}')
print(f'  AUC train = {metrics_a["AUC"]} | CV AUC = {metrics_a["CV_AUC"]}')
if PYGAM_OK and gam_gs is not None:
    print(f'Parte B (pyGAM):')
    print(f'  lambda   = {gam_gs.lam}')
    print(f'  AUC train = {metrics_b["AUC"]} | CV AUC = {metrics_b["CV_AUC"]}')
print('=' * 65)

## Kaggle Submission — AUC Test (Public Leaderboard)

Envía el CSV a `home-credit-default-risk` y recupera el AUC del Public LB (~30% del test).
Usa la Python API (`KaggleApi`) directamente — polling cada 10 s, máx 5 min.

> **Límite**: 5 submissions/día.


### Parte A — SplineTransformer + Logit L1


In [ ]:
from kaggle import KaggleApi
import time

COMPETITION = 'home-credit-default-risk'
_sub_path   = DATA_DIR / 'submission_08a_spline_logit.csv'
_msg = f"08a_spline_logit | train={metrics_a["AUC"]:.4f} | obj={study_a.best_value:.4f}"

def _as_str(x):
    return '' if x is None else str(x)

def _get_score(api, comp, file_name, message, poll=10, timeout=300):
    """Poll until public_score appears on the matching submission."""
    start = time.time()
    while time.time() - start < timeout:
        subs = api.competition_submissions(comp)
        matched = next(
            (s for s in subs
             if _as_str(getattr(s, 'file_name', None)) == file_name
             and _as_str(getattr(s, 'description', None)) == message),
            subs[0] if subs else None
        )
        if matched:
            pub     = _as_str(getattr(matched, 'public_score',  None))
            status  = _as_str(getattr(matched, 'status',        None))
            elapsed = int(time.time() - start)
            print(f'  [{elapsed:>3}s] status={status!r}  public_score={pub!r}')
            if pub and pub.lower() not in ('', 'none', 'null', '-'):
                priv = _as_str(getattr(matched, 'private_score', None))
                return float(pub), (float(priv) if priv and priv.lower() not in ('','none','null','-') else None)
        time.sleep(poll)
    return None, None

_api = KaggleApi()
_api.authenticate()

print(f'Enviando: {_msg}')
_api.competition_submit(file_name=str(_sub_path), message=_msg, competition=COMPETITION)
print('Esperando scoring (poll 10 s / máx 5 min)...')

public_auc, private_auc = _get_score(_api, COMPETITION, _sub_path.name, _msg)
print(f'\nAUC test  Public  LB : {public_auc}')
print(f'AUC test  Private LB : {private_auc}')


### Parte B — pyGAM


In [ ]:
if PYGAM_OK and gam_gs is not None:
    from kaggle import KaggleApi
    import time

    def _as_str(x): return '' if x is None else str(x)
    def _get_score(api, comp, file_name, message, poll=10, timeout=300):
        start = time.time()
        while time.time() - start < timeout:
            subs = api.competition_submissions(comp)
            matched = next(
                (s for s in subs
                 if _as_str(getattr(s, 'file_name', None)) == file_name
                 and _as_str(getattr(s, 'description', None)) == message),
                subs[0] if subs else None
            )
            if matched:
                pub = _as_str(getattr(matched, 'public_score', None))
                if pub and pub.lower() not in ('', 'none', 'null', '-'):
                    return float(pub)
            time.sleep(poll)
        return None, None

    _api2 = KaggleApi()
    _api2.authenticate()
    _msg_b = f'08b_pygam | train={metrics_b["AUC"]:.4f} | CV_GAM={gam_cv_mean:.4f}'
    _sub_b = DATA_DIR / 'submission_08b_pygam.csv'
    _api2.competition_submit(file_name=str(_sub_b), message=_msg_b,
                             competition='home-credit-default-risk')
    public_auc_b = _get_score(_api2, 'home-credit-default-risk', _sub_b.name, _msg_b)
    print(f'AUC test PB (pyGAM): {public_auc_b}')
else:
    print('pyGAM no disponible — submission B omitida.')
